# Task 1: Data acquisition

How to download NHL play-by-play data (2016-17 to 2023-24), cache it locally, and load it back.
All the logic lives in `ift6758/data/nhl.py`; this notebook is the walkthrough you can adapt for the blog post.

**Before running:** copy `.env.example` to `.env` (or `export NHL_DATA_DIR=...`) to choose where raw JSON is cached.
The default is `data/raw/`, which is git-ignored. Never commit the data.

In [ ]:
import json
from ift6758.data import (
    get_game, download_season, download_seasons, load_season,
    regular_season_game_id, playoff_game_id,
)

## 1. Game IDs

A game ID is `YYYY` + `TT` + `NNNN`: season start year, game type (`02` regular, `03` playoffs), then a game number.
For playoffs the last four digits are `0` + round + matchup + game number.

In [ ]:
print(regular_season_game_id(2016, 1))   # first game of the 2016-17 regular season
print(playoff_game_id(2022, 4, 1, 1))    # 2022-23 Stanley Cup Final, game 1 -> 2022030411

## 2. Fetch one game

`get_game` checks the cache first. The first call downloads and saves the raw JSON; later calls read from disk without any request.

In [ ]:
game = get_game("2022030411")
print(list(game.keys()))
print(len(game["plays"]), "events")
game["plays"][0]

## 3. Download a whole season

Regular season: counts up from game 1 and stops after a few consecutive 404s (season lengths differ).
Playoffs: walks round, matchup, game 1-7 and moves to the next series at the first missing game.
Re-running is free because everything is cached.

In [ ]:
season = download_season(2016)
{k: len(v) for k, v in season.items()}

## 4. Everything we need: 2016-17 to 2023-24

Slow the first time (be patient, someone else is paying for these requests), instant afterwards.

In [ ]:
# download_seasons(first=2016, last=2023)

## 5. Load from disk only

`load_season` never touches the network, which is what you want when building the tidy dataset.

In [ ]:
regular = load_season(2016, "regular")
playoffs = load_season(2016, "playoffs")
len(regular), len(playoffs)

## 6. Sanity checks

Compare per-season counts against the NHL's published game counts before trusting the data.

In [ ]:
for s in range(2016, 2024):
    print(s, len(load_season(s, "regular")), len(load_season(s, "playoffs")))